[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tensor-Reloaded/Deep-Learning-and-Generative-AI-2026/blob/main/Lab05/ObjectDetection.ipynb)

# Lab 05 · Turn an ImageNet classifier into a pedestrian detector

## Goal
Build a **small Fast R-CNN-style detector** in **120 minutes**. The ResNet-18 backbone and its ImageNet classification weights are supplied. **You implement the detection parts and train them on real bounding-box data**: image/box preprocessing, proposal assignment, a two-branch head, the multi-task loss, the training step, the epoch loop, and end-to-end fine-tuning of the last backbone block.

**Pipeline:** image → ResNet features → geometric region proposals → RoI pooling → your head → pedestrian scores + box corrections → NMS.

**Two required training experiments:** (A) transfer learning with a frozen backbone and a new detection head; (B) fine-tune `layer3` together with that head, recomputing features from images and using random horizontal flips. Compare both on validation, then evaluate the final model once on test.

This is a teaching adaptation of [Fast R-CNN](https://arxiv.org/abs/1504.08083), with its shared feature map and RoI pooling. We use ResNet-18 through `layer3`, fixed geometric proposals instead of selective search, and a class-agnostic box regressor. **There is no pretrained detector or RPN.** Faster R-CNN would additionally learn the proposals. Our image resize stretches to a square for simplicity; production pipelines usually preserve aspect ratio.

| Time | Work | Checkpoint |
|---|---|---|
| 0–10 min | Setup and pipeline recap | Download data and ImageNet weights |
| 10–25 min | TODO 1: resize image and boxes | Boxes still surround the people |
| 25–40 min | TODO 2: assign proposals | Region labels and matched objects |
| 40–55 min | TODO 3: build the detection head | Two correctly shaped outputs |
| 55–70 min | TODOs 4–5: loss and training step | Multi-task objective and optimization step |
| 70–85 min | TODO 6: epoch loop; train the new head | Frozen-backbone validation baseline |
| 85–110 min | TODO 7: end-to-end fine-tuning | Fine-tuned detection backbone and head |
| 110–120 min | Compare both experiments; test once; save | Predictions, metrics and explanations |

Work in pairs if useful. **Fill the seven numbered `TODO` blocks**; surrounding data, geometry, visualization and evaluation helpers are provided. This in-class exercise is separate from Homework 3 (Noisy CIFAR-100).

## Setup
1. Open in Colab and choose **File → Save a copy in Drive**.
2. Choose **Runtime → Change runtime type → T4 GPU** (if offered), then connect. Free GPU availability is not guaranteed. This small frozen-backbone exercise also supports CPU.
3. Run cells in order. Stop at each TODO and implement it before continuing. A `NotImplementedError` is intentional, not an installation failure.

Colab normally includes all dependencies below; **do not reinstall torch/torchvision in a working Colab runtime**. If imports fail locally, install a compatible pair from [PyTorch](https://pytorch.org/get-started/locally/) plus `numpy matplotlib pillow tqdm`.
The dataset (~51 MB) and ResNet-18 weights (~45 MB) download automatically. No Kaggle token, Drive mount or repository clone is required. Downloads need an internet connection; a runtime reset clears local data and model state.

In [ ]:
import copy, os, random, time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.ops import box_iou, roi_pool, nms, clip_boxes_to_image, remove_small_boxes
from torchvision.datasets.utils import download_and_extract_archive
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# CPU is deliberate on Macs: compiled RoI pooling is not supported on every MPS version.
if device.type == 'cpu':
    torch.set_num_threads(min(4, os.cpu_count() or 1))
SIZE = 256                    # ResNet feature stride is 16
POOL_SIZE = 3                 # each proposal becomes [256, 3, 3]
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 5
LR = 1e-3
BACKBONE_LR = 1e-4
FINETUNE_HEAD_LR = 3e-4
ROIS_PER_IMAGE = 64
IMAGES_PER_STEP = 4
SCORE_THRESHOLD = 0.7
NMS_THRESHOLD = 0.3
DATA_DIR = Path(os.environ.get('CARN05_DATA', './data'))
DATA_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__, '| device:', device)

### Data: Penn-Fudan pedestrians
[Dataset and attribution](https://www.cis.upenn.edu/~jshi/ped_html/) · [Official torchvision tutorial using this dataset](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html).
Each instance-mask value identifies one person; zero is background. We derive one bounding box per person and discard the masks afterwards. Class IDs: **0 = background, 1 = pedestrian**. All boxes use **`[x1, y1, x2, y2]`**, with exclusive upper edges; coordinates are in pixels, not normalized to [0,1].

The seeded split is **120 train / 30 validation / 20 test** images. It is an educational image-level split of a small dataset with similar scenes, not a robust deployment benchmark. Only train images provide gradients; validation is for inspection/tuning, and test is evaluated once at the end.

In [ ]:
root = DATA_DIR / 'PennFudanPed'
if not root.exists():
    download_and_extract_archive(
        'https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip',
        str(DATA_DIR), filename='PennFudanPed.zip')
paths = sorted((root / 'PNGImages').glob('*.png'))
order = np.random.default_rng(SEED).permutation(len(paths))
train_ids, val_ids, test_ids = order[:120].tolist(), order[120:150].tolist(), order[150:].tolist()

def read_raw(index):
    path = paths[index]
    image = Image.open(path).convert('RGB')
    mask = np.array(Image.open(root / 'PedMasks' / (path.stem + '_mask.png')))
    boxes = []
    for instance in np.unique(mask):
        if instance == 0:
            continue
        ys, xs = np.where(mask == instance)
        boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
    return image, torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)

weights = ResNet18_Weights.IMAGENET1K_V1
classifier = resnet18(weights=weights)
# conv1 ... layer3. Remove layer4, average pooling and the 1000-class classifier.
backbone = nn.Sequential(*list(classifier.children())[:-3]).to(device).eval()
backbone.requires_grad_(False)
imagenet_backbone_state = copy.deepcopy(backbone.state_dict())
del classifier
MEAN = torch.tensor(weights.transforms().mean)[:, None, None]
STD = torch.tensor(weights.transforms().std)[:, None, None]
with torch.no_grad():
    feature_shape = backbone(torch.zeros(1, 3, SIZE, SIZE, device=device)).shape
print('Frozen pretrained backbone:', tuple(feature_shape), '| split:', len(train_ids), len(val_ids), len(test_ids))

## Steps
### 1 · Preprocessing (TODO 1, 15 minutes)
Resize the **whole** image to `(SIZE, SIZE)` and transform the boxes by the same geometry. Do not use the classification center crop: it could remove objects while leaving their labels behind.

If the original dimensions are `W × H`, then `sx = SIZE/W`, `sy = SIZE/H`. Multiply the four coordinates by `[sx, sy, sx, sy]`. Convert RGB to a float tensor in [0,1], resize with bilinear interpolation, then normalize by ImageNet mean/std. Return `(normalized_image, resized_boxes)`; do not change the input box tensor.

Useful APIs: `TF.to_tensor`, `TF.resize(..., [SIZE, SIZE], interpolation=InterpolationMode.BILINEAR, antialias=True)`.

In [ ]:
def preprocess(image, boxes):
    # TODO 1: image [3,SIZE,SIZE], boxes [N,4] in resized-image pixel coordinates.
    raise NotImplementedError('TODO 1: resize both the image and its boxes, then normalize')

In [ ]:
def draw_boxes(ax, image, boxes, title, scores=None, color='#0072B2', style='-'):
    if isinstance(image, torch.Tensor):
        image = (image.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(image)
    for k, (x1, y1, x2, y2) in enumerate(boxes.cpu().tolist()):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                               edgecolor=color, linewidth=2, linestyle=style))
        if scores is not None:
            ax.text(x1+2, y1+2, f'{float(scores[k]):.2f}', color='white', fontsize=9, va='top',
                    bbox=dict(facecolor=color, edgecolor='none', pad=1))
    ax.set_title(title); ax.axis('off')

raw_image, raw_boxes = read_raw(train_ids[0])
image, gt_boxes = preprocess(raw_image, raw_boxes)
fig, axes = plt.subplots(1, 2, figsize=(10, 5), layout='constrained')
draw_boxes(axes[0], raw_image, raw_boxes, 'Original image + annotation')
draw_boxes(axes[1], image, gt_boxes, 'Resized image + transformed annotation')
plt.show()

### 2 · Proposals and training targets (TODO 2, 15 minutes)
Our **provided, image-independent grid** proposes tall rectangles at several scales and positions. It never reads annotations. This substitutes for selective search; it is not an RPN. Inspect proposal recall: how many true objects have at least one candidate with IoU ≥ 0.5?

For training only, we append the true boxes to the proposals so each object can supply a positive example (standard detector training practice). These exact boxes have zero regression targets; other positive grid proposals teach nonzero corrections. **Validation/test predictions must never receive or append ground-truth boxes.**

Implement `assign_targets`:
1. Find each proposal's largest IoU and the index of the corresponding true box.
2. Assign **1 if IoU ≥ 0.5**, **0 if IoU < 0.3**, **−1 otherwise** (ignore ambiguous examples).
3. Return labels `[R]` (long) and the corresponding true box `[R,4]` for each proposal. Matched boxes for background/ignored proposals are placeholders and must not enter box loss.
4. If there are no objects, label every proposal background and return zero matched boxes.

`box_iou(proposals, gt)` returns an `[R,G]` matrix. Use `.max(dim=1)` when `G > 0`. Balanced sampling is supplied: at most 16 positives and up to 48+ negatives per image, without ignored regions.

In [ ]:
def grid_proposals(size=SIZE):
    boxes = []
    for height in [48, 80, 112, 160, 208, 256]:
        for ratio in [0.3, 0.45, 0.65]:  # width / height
            width = round(height * ratio)
            # Include right/bottom edges even when the step doesn't land on them.
            xs = sorted(set(range(0, size-width+1, 16)) | {size-width})
            ys = sorted(set(range(0, size-height+1, 16)) | {size-height})
            boxes.extend([[x, y, x+width, y+height] for y in ys for x in xs])
    return torch.tensor(boxes, dtype=torch.float32)

proposals = grid_proposals()

def proposal_recall(indices, threshold=0.5):
    hits = total = 0
    for index in indices:
        raw, boxes = read_raw(index)
        _, boxes = preprocess(raw, boxes)
        hits += (box_iou(proposals, boxes).max(dim=0).values >= threshold).sum().item()
        total += len(boxes)
    return hits / max(total, 1)

print('Proposals per image:', len(proposals))
print('Validation proposal recall @ IoU 0.5:', round(proposal_recall(val_ids), 3))
# Show the five best grid proposals for one annotated person, only for explanation.
best = box_iou(proposals, gt_boxes[:1])[:, 0].topk(5).indices
fig, ax = plt.subplots(figsize=(6, 5), layout='constrained')
draw_boxes(ax, image, proposals[best], 'Five nearby proposals (blue); true box (orange dashed)')
draw_boxes(ax, image, gt_boxes[:1], ax.get_title(), color='#D55E00', style='--')
plt.show()

In [ ]:
def assign_targets(proposals, gt):
    # TODO 2: labels [R] int64 and matched boxes [R,4], on the proposals' device.
    raise NotImplementedError('TODO 2: match IoUs and assign positive/background/ignore labels')

### Box regression and feature caching — provided helpers
For proposal center `(px,py)` and size `(pw,ph)`, the target for a matched true box is:

`tx=(gx−px)/pw`, `ty=(gy−py)/ph`, `tw=log(gw/pw)`, `th=log(gh/ph)`.

The head predicts these **relative corrections**, not absolute corners. Decoding reverses the formulas. A zero correction leaves the proposal unchanged. We compute/cache each image's feature map once because the backbone is frozen and in evaluation mode. This cache is only valid for experiment A. In the required experiment B you will recompute features inside the gradient-tracked forward pass, because `layer3` will change.

In [ ]:
def encode_boxes(proposals, targets):
    pwh = proposals[:, 2:] - proposals[:, :2]
    pxy = (proposals[:, 2:] + proposals[:, :2]) / 2
    gwh = targets[:, 2:] - targets[:, :2]
    gxy = (targets[:, 2:] + targets[:, :2]) / 2
    return torch.cat(((gxy - pxy) / pwh, torch.log(gwh / pwh)), dim=1)

def decode_boxes(proposals, deltas):
    pwh = proposals[:, 2:] - proposals[:, :2]
    pxy = (proposals[:, 2:] + proposals[:, :2]) / 2
    xy = pxy + deltas[:, :2] * pwh
    wh = pwh * deltas[:, 2:].clamp(-4, 4).exp()  # guard extreme predictions
    return torch.cat((xy - wh / 2, xy + wh / 2), dim=1)


@torch.no_grad()
def cache_images(indices):
    cache = []
    for index in tqdm(indices, desc='Frozen features', leave=False):
        raw, boxes = read_raw(index)
        x, boxes = preprocess(raw, boxes)
        cache.append({'index': index, 'image': x, 'boxes': boxes,
                      'features': backbone(x[None].to(device)).cpu()})
    return cache

start = time.perf_counter()
train_cache = cache_images(train_ids)
val_cache = cache_images(val_ids)
# Test features are created only in the final evaluation section.
print(f'Cached {len(train_cache)+len(val_cache)} images in {time.perf_counter()-start:.1f}s.')

def sample_training_regions(record):
    candidates = torch.cat((proposals, record['boxes']))  # TRAIN ONLY
    labels, matched = assign_targets(candidates, record['boxes'])
    pos = torch.where(labels == 1)[0]
    neg = torch.where(labels == 0)[0]
    pos = pos[torch.randperm(len(pos))[:ROIS_PER_IMAGE // 4]]
    neg = neg[torch.randperm(len(neg))[:ROIS_PER_IMAGE - len(pos)]]
    keep = torch.cat((pos, neg))
    targets = torch.zeros(len(keep), 4)
    if len(pos):
        targets[:len(pos)] = encode_boxes(candidates[pos], matched[pos])
    return candidates[keep], labels[keep], targets

@torch.no_grad()
def make_training_batch(records):
    pooled, labels, targets = [], [], []
    for record in records:
        boxes, lab, target = sample_training_regions(record)
        # Boxes are in IMAGE coordinates. spatial_scale=1/16 maps to the feature map.
        features = record['features'].to(device)
        pooled.append(roi_pool(features, [boxes.to(device)], (POOL_SIZE, POOL_SIZE), spatial_scale=1/16))
        labels.append(lab.to(device)); targets.append(target.to(device))
    return torch.cat(pooled), torch.cat(labels), torch.cat(targets)

batch = make_training_batch(train_cache[:IMAGES_PER_STEP])
print('Pooled regions, labels, targets:', [tuple(x.shape) for x in batch])

### 3 · The detection head (TODO 3, 15 minutes)
Implement a shared MLP followed by two separate linear branches:

| Part | Input → output | Activation |
|---|---|---|
| Shared | flatten `[R,256,3,3]` → Linear(2304,128) | ReLU |
| Classifier | 128 → 2 | **none**: raw background/person logits |
| Box regressor | 128 → 4 | **none**: `(tx,ty,tw,th)` can be negative |

Return `(class_logits, box_deltas)`. The box branch is class-agnostic because there is only one foreground class. Use `nn.Sequential`, `nn.Flatten`, `nn.Linear`, `nn.ReLU`. Initialize the regressor weights and bias to zero so initial boxes equal proposals. Do not apply softmax before cross-entropy.

In [ ]:
class DetectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 3a: shared MLP, classifier and box_regressor. Zero-initialize box_regressor.
        raise NotImplementedError('TODO 3a: construct the head')

    def forward(self, pooled):
        # TODO 3b: return raw class logits [R,2] and box deltas [R,4].
        raise NotImplementedError('TODO 3b: forward through the two branches')

In [ ]:
torch.manual_seed(SEED)
head = DetectionHead().to(device)

### 4 · Two losses, one objective (TODO 4, 8 minutes)
Implement `L = L_class + L_box`:
- `L_class`: `F.cross_entropy(logits, labels)` over **all sampled regions**, including background.
- `L_box`: `F.smooth_l1_loss(deltas[pos], targets[pos], beta=1.0)` over **positive regions only**, with mean reduction. Here `pos = labels == 1`.
- If there are no positives, use `deltas.sum() * 0` for a differentiable zero regression loss.

Return `(total_loss, classification_loss, regression_loss)`. This lab uses unweighted mean losses for readability; the original paper's normalization/class-specific formulation differs. Background regions have no meaningful box target. Ignored regions were already removed by the sampler.

In [ ]:
def detection_loss(logits, deltas, labels, targets):
    # TODO 4: CE over all sampled regions; Smooth L1 on foreground only.
    raise NotImplementedError('TODO 4: implement the multi-task loss')

### 5 · Training step (TODO 5, 7 minutes)
Implement the part that actually learns. Balanced region batches are provided. You will write the outer epoch loop in TODO 6.

In order: put the head in training mode, clear old gradients (`optimizer.zero_grad(set_to_none=True)`), forward pass, compute the two losses, `backward()`, then `optimizer.step()`. Return three **Python floats**, not tensors retaining the computation graph. Do not put the head forward/backward inside `torch.no_grad()`; only the frozen backbone/cache uses that context.

In [ ]:
def train_step(model, batch, optimizer):
    pooled, labels, targets = batch
    # TODO 5: clear gradients, forward, loss, backward, update; return 3 floats.
    raise NotImplementedError('TODO 5: implement one optimization step')

### Inference and evaluation — supplied
Inference receives **features and the geometric grid only**. The head scores candidates, decodes box corrections, clips to the image, drops small/low-score boxes, then applies NMS. Ground truth is used only afterwards for evaluation/visual comparison.

Precision and recall below use **one-to-one greedy matching at IoU ≥ 0.5**, detections processed by descending score. A second detection of the same person is a false positive. These are fixed-threshold precision/recall, **not COCO mAP**. NMS IoU=0.3, matching IoU=0.5 and score cutoff=0.7 serve different purposes.

In [ ]:
@torch.no_grad()
def detect(model, features, score_threshold=SCORE_THRESHOLD, nms_threshold=NMS_THRESHOLD, refine=True):
    model.eval()
    features = features.to(device)
    all_boxes, all_scores = [], []
    for candidates in proposals.split(256):
        candidates = candidates.to(device)
        pooled = roi_pool(features, [candidates], (POOL_SIZE, POOL_SIZE), spatial_scale=1/16)
        logits, deltas = model(pooled)
        scores = logits.softmax(dim=1)[:, 1]
        boxes = decode_boxes(candidates, deltas) if refine else candidates
        boxes = clip_boxes_to_image(boxes, (SIZE, SIZE))
        keep = remove_small_boxes(boxes, min_size=4)
        keep = keep[scores[keep] >= score_threshold]
        all_boxes.append(boxes[keep]); all_scores.append(scores[keep])
    boxes, scores = torch.cat(all_boxes), torch.cat(all_scores)
    keep = nms(boxes, scores, nms_threshold)[:20]
    return boxes[keep].cpu(), scores[keep].cpu()

def match_counts(boxes, scores, gt, iou_threshold=0.5):
    used = set()
    true_positive = 0
    for index in scores.argsort(descending=True).tolist():
        available = [j for j in range(len(gt)) if j not in used]
        if not available:
            continue
        ious = box_iou(boxes[index:index+1], gt[available])[0]
        value, position = ious.max(dim=0)
        if value >= iou_threshold:
            used.add(available[position.item()]); true_positive += 1
    return true_positive, len(boxes)-true_positive, len(gt)-true_positive

@torch.no_grad()
def evaluate(model, cache):
    counts = np.zeros(3, dtype=int)
    for record in cache:
        boxes, scores = detect(model, record['features'])
        counts += np.array(match_counts(boxes, scores, record['boxes']))
    tp, fp, fn = counts.tolist()
    return {'precision': tp / max(tp+fp, 1), 'recall': tp / max(tp+fn, 1), 'TP': tp, 'FP': fp, 'FN': fn}

initial_state = copy.deepcopy(head.state_dict())
baseline = evaluate(head, val_cache)
print('Untrained head, validation:', baseline)

### 6 · Write the epoch loop and train the head (TODO 6, 15 minutes)
Experiment A: frozen-backbone transfer learning. The ImageNet classifier has become a detector through your new trainable head and bounding-box supervision.

Implement `train_epochs` so it works for **both experiments**:
1. At each epoch, shuffle record indices using `torch.randperm`.
2. Iterate in batches of `IMAGES_PER_STEP`; include the last partial batch.
3. Call `batch_builder(records)` then `step_function(model, batch, optimizer)`.
4. Collect the three losses and append their mean per epoch to `history`.
5. Print one compact progress line per epoch and return `np.array(history)` of shape `[epochs,3]`.

Do not recreate/reset the optimizer each epoch. The caller supplies a fresh optimizer at the start of an experiment. The loop should work with either cached RoIs (A) or full images (B).

In [ ]:
def train_epochs(model, records, optimizer, epochs, batch_builder, step_function):
    # TODO 6: shuffled epochs, minibatches, optimizer steps, three mean losses per epoch.
    raise NotImplementedError('TODO 6: implement the training loop')

In [ ]:
# Re-running this cell restarts experiment A. Keep the supplied backbone frozen here.
backbone.load_state_dict(imagenet_backbone_state)
backbone.requires_grad_(False); backbone.eval()
head.load_state_dict(initial_state)
torch.manual_seed(SEED)
optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
start = time.perf_counter()
head_history = train_epochs(head, train_cache, optimizer, HEAD_EPOCHS, make_training_batch, train_step)
head_seconds = time.perf_counter()-start
head_only_state = copy.deepcopy(head.state_dict())
head_only_metrics = evaluate(head, val_cache)
print(f'Experiment A: {head_seconds:.1f}s on {device}; validation:', head_only_metrics)

@torch.no_grad()
def validation_predictions(model, records):
    return [detect(model, record['features']) for record in records]
head_only_predictions = validation_predictions(head, val_cache[:3])

### 7 · Fine-tune the backbone on detection images (TODO 7, 25 minutes)
Experiment B: **real end-to-end fine-tuning** of `layer3` and the detection head. Earlier layers stay frozen. This starts from experiment A's head; it is not training from scratch.

The supplied batch builder loads the resized images, sometimes flips an image **and its boxes**, then samples proposals/targets. It returns `(images, region_boxes, labels, targets)`. `region_boxes` is one tensor per image. The RoI pooling output concatenates regions in that same order.

Implement `finetune_step`:
1. `model.train()` and `backbone.eval()`. **Evaluation mode does not disable gradients.** We hold pretrained BatchNorm running statistics fixed because batches are tiny.
2. Clear gradients of the optimizer (it owns both `layer3` and the head).
3. Compute `features = backbone(images)` **without `no_grad` and without `.detach()`**.
4. Call `roi_pool(features, region_boxes, (POOL_SIZE, POOL_SIZE), spatial_scale=1/16)`.
5. Run your head and multi-task loss, backpropagate, update, return three Python floats.

Use the epoch loop you already wrote. We use a smaller learning rate for pretrained `layer3` than for the new head. Reusing cached features in this stage would prevent transfer learning from updating the backbone.

In [ ]:
def make_image_batch(records, augment=True):
    images, region_boxes, all_labels, all_targets = [], [], [], []
    for record in records:
        image, boxes = record['image'], record['boxes']
        if augment and torch.rand(()) < 0.5:
            image = image.flip(-1)
            boxes = boxes.clone()
            boxes[:, 0], boxes[:, 2] = SIZE - record['boxes'][:, 2], SIZE - record['boxes'][:, 0]
        temporary = {'image': image, 'boxes': boxes}
        regions, labels, targets = sample_training_regions(temporary)
        images.append(image); region_boxes.append(regions.to(device))
        all_labels.append(labels); all_targets.append(targets)
    return (torch.stack(images).to(device), region_boxes,
            torch.cat(all_labels).to(device), torch.cat(all_targets).to(device))

# The ImageNet backbone snapshot was saved at setup, before either experiment.
backbone.requires_grad_(False)
backbone[-1].requires_grad_(True)  # layer3
backbone.eval()

def make_finetune_optimizer(model):
    return torch.optim.AdamW([
        {'params': backbone[-1].parameters(), 'lr': BACKBONE_LR},
        {'params': model.parameters(), 'lr': FINETUNE_HEAD_LR},
    ], weight_decay=1e-4)

print('Fine-tuning backbone parameters:', sum(p.numel() for p in backbone.parameters() if p.requires_grad))
print('Head parameters:', sum(p.numel() for p in head.parameters()))

In [ ]:
def finetune_step(model, batch, optimizer):
    images, region_boxes, labels, targets = batch
    # TODO 7: image -> trainable backbone -> RoI pooling -> head -> loss -> backward -> step.
    # Do not use the cached features here. Keep BatchNorm statistics fixed via backbone.eval().
    raise NotImplementedError('TODO 7: fine-tune layer3 and the detection head together')

In [ ]:
# Start experiment B from A. Re-run from setup to restart the entire two-stage experiment.
backbone.load_state_dict(imagenet_backbone_state)
head.load_state_dict(head_only_state)
backbone.requires_grad_(False); backbone[-1].requires_grad_(True); backbone.eval()
torch.manual_seed(SEED)
finetune_optimizer = make_finetune_optimizer(head)
start = time.perf_counter()
finetune_history = train_epochs(head, train_cache, finetune_optimizer, FINETUNE_EPOCHS,
                                make_image_batch, finetune_step)
finetune_seconds = time.perf_counter()-start
# Backbone weights changed: discard stale validation features before inference.
val_cache_finetuned = cache_images(val_ids)
finetuned_metrics = evaluate(head, val_cache_finetuned)
print(f'Experiment B: {finetune_seconds:.1f}s on {device}; validation:', finetuned_metrics)
print('Validation comparison (same split and thresholds):')
for name, metrics in [('Untrained head', baseline), ('A: head only', head_only_metrics), ('B: layer3 + head', finetuned_metrics)]:
    print(name, metrics)

fig, axes = plt.subplots(1, 2, figsize=(12,4), layout='constrained')
for ax, history, title in zip(axes, [head_history, finetune_history], ['A: head transfer learning', 'B: detection fine-tuning']):
    for i, (name, color, style) in enumerate([('Total','#333333','-'), ('Classification','#0072B2','--'), ('Box','#D55E00',':')]):
        ax.plot(range(1, len(history)+1), history[:,i], label=name, color=color, linestyle=style)
    ax.set(xlabel='Epoch', ylabel='Mean training loss', title=title, ylim=(0,None))
    ax.legend()
plt.show()

## Results · What did the detector learn?
The display uses the **same first three validation images** for both experiments, not hand-picked successes. Orange dashed boxes are annotations; blue solid boxes are predictions. Compare head-only transfer learning with end-to-end fine-tuning. Inspect misses, duplicate detections and localization. Both use box correction and NMS; neither prediction receives ground truth. Fine-tuning is not guaranteed to improve every image or metric.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 12), layout='constrained')
for row, record in enumerate(val_cache_finetuned[:3]):
    draw_boxes(axes[row,0], record['image'], record['boxes'], 'Annotation', color='#D55E00', style='--')
    predictions = [head_only_predictions[row], detect(head, record['features'])]
    for col, ((boxes, scores), title) in enumerate(zip(predictions, ['A: head only', 'B: fine-tuned layer3 + head']), start=1):
        draw_boxes(axes[row,col], record['image'], boxes, title, scores=scores)
        draw_boxes(axes[row,col], record['image'], record['boxes'], title, color='#D55E00', style='--')
plt.show()

### Final held-out evaluation and hand-in
Use validation to choose any thresholds or training settings first. Then run the test cell once and report **all** its precision/recall and TP/FP/FN results, including failures. Do not optimize repeatedly on test.

Hand in your completed notebook with: the seven implementations, loss curves for both experiments, validation comparison, final test metrics, and short answers below. Save your Colab copy; the saved checkpoint contains the fine-tuned backbone and head, plus split/configuration metadata. It does not contain the dataset.

In [ ]:
test_cache = cache_images(test_ids)
test_metrics = evaluate(head, test_cache)
print('Final test (20 held-out images), IoU >= 0.5:', test_metrics)
checkpoint = Path('lab05_finetuned_detector.pt')
torch.save({'head': {k: v.cpu() for k,v in head.state_dict().items()},
            'backbone': {k: v.cpu() for k,v in backbone.state_dict().items()},
            'backbone_weights': 'ResNet18_Weights.IMAGENET1K_V1',
            'size': SIZE, 'pool_size': POOL_SIZE, 'seed': SEED,
            'score_threshold': SCORE_THRESHOLD, 'nms_threshold': NMS_THRESHOLD,
            'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids}, checkpoint)
print('Saved', checkpoint, '— use Colab Files to download if wanted.')

### Explain in your own words
1. Why must image resize also change the box coordinates? What would a center crop break?
2. Why does box loss ignore background? What happens when a batch has no positives?
3. Why is `spatial_scale=1/16`? Why does caching become wrong when the backbone is unfrozen?
4. Why is this a Fast R-CNN-style detector, rather than Faster R-CNN? What would an RPN add?
5. Compare A versus B: validation precision/recall, TP/FP/FN, runtime, and one failure case. Did fine-tuning improve every metric/image?
6. Why can `backbone.eval()` coexist with learning its weights? Why use different learning rates for the pretrained block and the new head?

**Your answers:** replace this paragraph.

## Next steps · optional, after completing the core lab
- Re-run a validation example with NMS thresholds 0.1 and 0.7. Describe duplicate detections versus missed nearby people.
- Implement `encode_boxes` yourself from the formulas.
- Replace the geometric proposals with selective search and compare proposal recall and runtime.
- Compare head-only training for 10 epochs against 5 head-only + 5 fine-tuning epochs: this better controls training duration. A versus B above also changes augmentation and training steps, so it does not isolate the effect of unfreezing.
- Replace square resize with aspect-ratio-preserving letterboxing and transform the boxes accordingly.
- Implement an RPN as a longer project: objectness, anchor matching, box deltas and joint training.

### Sources
- [Fast R-CNN — paper](https://arxiv.org/abs/1504.08083) · [PDF, architecture and multi-task loss](https://arxiv.org/pdf/1504.08083)
- [ResNet — paper](https://arxiv.org/abs/1512.03385) · [torchvision ResNet-18 weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [R-CNN](https://arxiv.org/abs/1311.2524) · [Faster R-CNN](https://arxiv.org/abs/1506.01497)
- [RoI pooling API](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.roi_pool.html) · [NMS API](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.nms.html)
- [Penn-Fudan dataset](https://www.cis.upenn.edu/~jshi/ped_html/) · [PyTorch dataset tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)
- [Previously presented ML@B Object Detection slides](https://docs.google.com/presentation/d/1E4DPhSEGXd9ZnIGzbhnpaPxfqzStpL3xtz5Y0jz5P98/edit#slide=id.g23ab59ea0ea_0_661) (Fast R-CNN: slides 18–19)
- [Colab resource availability](https://research.google.com/colaboratory/faq.html)